<a href="https://colab.research.google.com/github/kanakamvasundhara/research-paper-rag/blob/main/research_paper_qa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pypdf
!pip install faiss-cpu
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline

In [ ]:
pdf_file = "/content/nlp_application.pdf"

In [ ]:
reader = PdfReader(pdf_file)

pages = []

for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text:
        pages.append(text)

print("Number of pages:", len(pages))
print("Text extracted successfully")

In [ ]:
chunks = []
chunk_size = 500
overlap = 100

for page_no, text in enumerate(pages, start=1):
    words = text.split()

    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])

        if len(chunk.strip()) > 50:
            chunks.append({
                "text": chunk,
                "page": page_no
            })

print("Total chunks:", len(chunks))

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype("float32"))

print("Vectors stored:", index.ntotal)

In [ ]:
generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=250
)

In [ ]:
def ask_question(question, k=3):

    query_embedding = model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, k)

    retrieved = []

    for idx in indices[0]:
        retrieved.append(chunks[idx])

    context = "\n\n".join(
        f"[Page {item['page']}]\n{item['text']}"
        for item in retrieved
    )

    prompt = f"""
Answer the question using ONLY the information given in the context.

If the answer is not available in the context, say:
"Information not found in the research paper."

Give a short and clear answer.

Context:
{context}

Question:
{question}

Answer:
"""

    answer = generator(prompt)[0]["generated_text"]

    print("\nANSWER:")
    print(answer)

    print("\nSOURCES:")
    for item in retrieved:
        print(f"- Page {item['page']}")

    return answer

In [1]:
question = input("Enter your question: ")

ask_question(question)

KeyboardInterrupt: Interrupted by user